In [1]:
# --- TELEPÍTÉS ---
! nvidia-smi
! sudo apt-get install zstd -y
! curl -fsSL https://ollama.com/install.sh | sh
! pip install -qq pyngrok ollama

# --- KONFIGURÁCIÓ ---
# Ide írd be a tokenedet
AUTH_TOKEN = "2sUqYYhi9BuVxa2zJtzAdHVI9eS_6rxXaeqe4o8JTPtMPtsBf"
# Javaslat: Ha hibát dob a domainre, vedd ki a domain paramétert teszteléshez
NGROK_DOMAIN = "intimate-polecat-adjusted.ngrok-free.app"
NGROK_PORT = "11434"

import subprocess
import os
import time
from pyngrok import ngrok
import ollama

# --- TAKARÍTÁS (FONTOS!) ---
print("🧹 Takarítás: Régi folyamatok leállítása...")
os.system("pkill -f ngrok")
os.system("pkill -f ollama")
ngrok.kill()
time.sleep(2)  # Kis pihenő a rendszernek


def start_ollama_server() -> None:
    """Starts the Ollama server with CORS enabled."""
    print("🚀 Ollama szerver indítása...")
    env = os.environ.copy()
    env["OLLAMA_ORIGINS"] = "*"
    env["OLLAMA_HOST"] = "0.0.0.0:11434"
    # stdout=subprocess.DEVNULL elrejti a szemetet, ha nem kell a log
    subprocess.Popen(["ollama", "serve"], env=env)

    # Várunk, hogy tényleg elinduljon
    max_retries = 10
    for i in range(max_retries):
        try:
            # Próbálunk egy üres kérést küldeni, hogy él-e a port
            subprocess.run(["curl", "-s", "http://127.0.0.1:11434"], check=True)
            print("✅ Ollama szerver fut és elérhető!")
            return
        except:
            print(f"⏳ Várakozás az Ollama-ra... ({i + 1}/{max_retries})")
            time.sleep(2)
    print("⚠️ FIGYELEM: Az Ollama lassan indul, vagy hiba történt.")


def setup_ngrok_tunnel(port: str) -> ngrok.NgrokTunnel:
    if not AUTH_TOKEN:
        raise RuntimeError("AUTH_TOKEN is not set.")

    print("🔗 Ngrok tunnel építása...")
    ngrok.set_auth_token(AUTH_TOKEN)

    try:
        # Próbáljuk meg a fix domainnel
        tunnel = ngrok.connect(port, domain=NGROK_DOMAIN)
    except Exception as e:
        print(f"⚠️ Nem sikerült a fix domain, random URL generálása... (Hiba: {e})")
        # Ha a domain foglalt vagy hibás, indítsunk simát
        tunnel = ngrok.connect(port)

    print(f"🎉 SIKER! Ngrok URL: {tunnel.public_url}")
    return tunnel


# --- FŐ FOLYAMAT ---

start_ollama_server()

# Nem kell külön check_ollama_port, mert a start fv-be beépítettem az ellenőrzést

ngrok_tunnel = setup_ngrok_tunnel(NGROK_PORT)

# Kliens beállítása
client = ollama.Client(host=ngrok_tunnel.public_url)

print("📥 Modell letöltése (ez eltarthat egy darabig)...")
# subprocess.run(["ollama", "pull", "deepseek-coder-v2:16b"], check=True) # Vagy amit használni akarsz
subprocess.run(["ollama", "pull", "tom_himanen/deepseek-r1-roo-cline-tools:7b"], check=True)
print("✅ Kész! Mehet a használat.")

Sat Jan 24 09:56:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             30W /   70W |   12084MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----